# Phase 1: RAG Evaluation — The Vibe Check Fallacy


## 1. Imports and Configuration


In [1]:
#@title Install dependencies
%pip install -q anthropic chromadb sentence-transformers pandas numpy matplotlib langchain langchain-community langchain-chroma langchain-huggingface
print('✓ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/23

In [2]:
#@title Configuration { display-mode: "form" }
import os
import pandas as pd
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

ANTHROPIC_MODEL = "claude-haiku-4-5-20251001"

print("✓ Configuration complete")
print("Model:", ANTHROPIC_MODEL)

Enter your Anthropic API key: ··········
✓ Configuration complete
Model: claude-haiku-4-5-20251001


## 2. Synthetic Corpus — TechCorp Pro Billing FAQ

In [3]:
# Synthetic document corpus — TechCorp Pro Subscription & Billing FAQ
CORPUS_DOCS = [
    {
        "doc_id": "billing_cycle",
        "content": (
            "TechCorp Pro offers two billing cycles: monthly and annual. "
            "Monthly subscribers are billed on the same calendar date each month. "
            "For example, if you subscribed on the 15th, you are billed on the 15th of every month. "
            "Annual subscribers are billed once per year on their subscription anniversary date "
            "and receive a 20% discount compared to the monthly rate. "
            "If a billing date falls on a weekend or holiday, the charge is processed on the next business day."
        ),
        "metadata": {"source": "billing_faq", "section": "billing_cycle"},
    },
    {
        "doc_id": "cancellation_policy",
        "content": (
            "Subscribers may cancel at any time through the Account Settings page under the Subscription tab. "
            "After cancellation, the subscription remains active until the end of the current paid billing period. "
            "There are no cancellation fees or penalties. "
            "Cancelled accounts retain read-only access for 30 days after the billing period ends to allow data export."
        ),
        "metadata": {"source": "billing_faq", "section": "cancellation"},
    },
    {
        "doc_id": "refund_policy",
        "content": (
            "TechCorp Pro offers a 14-day money-back guarantee for new subscriptions only. "
            "Refund requests must be submitted within 14 days of the initial purchase date via the Help Center. "
            "Renewal charges are not eligible for refunds once the billing period has started. "
            "Refunds are processed back to the original payment method within 5 to 7 business days."
        ),
        "metadata": {"source": "billing_faq", "section": "refunds"},
    },
    {
        "doc_id": "plan_changes",
        "content": (
            "Subscribers can upgrade their plan at any time from the Account Settings page. "
            "Upgrades take effect immediately; the subscriber is charged a prorated amount for the remainder of the current billing cycle. "
            "Downgrades take effect at the start of the next billing cycle; no prorated credit is issued for downgrades. "
            "Switching from monthly to annual billing can be done at any time and includes a prorated credit for unused monthly days."
        ),
        "metadata": {"source": "billing_faq", "section": "plan_changes"},
    },
    {
        "doc_id": "payment_methods",
        "content": (
            "TechCorp Pro accepts Visa, Mastercard, American Express, and Discover credit and debit cards. "
            "PayPal is also supported as a payment method. "
            "Bank transfers (ACH) are available for annual subscribers on the Business or Enterprise plan only. "
            "Cryptocurrency payments are not accepted. "
            "Payment information is stored securely and is never shared with third parties."
        ),
        "metadata": {"source": "billing_faq", "section": "payment_methods"},
    },
]

print(f"✓ Synthetic corpus created: {len(CORPUS_DOCS)} documents")
for doc in CORPUS_DOCS:
    print(f"  [{doc['doc_id']}]: {len(doc['content'])} chars")

✓ Synthetic corpus created: 5 documents
  [billing_cycle]: 454 chars
  [cancellation_policy]: 350 chars
  [refund_policy]: 345 chars
  [plan_changes]: 433 chars
  [payment_methods]: 359 chars


## 3. Split Documents


In [4]:
# Each document is used as a single chunk (documents are short and self-contained)
chunks = [
    {
        "id": doc["doc_id"],
        "content": doc["content"],
        "metadata": doc["metadata"],
    }
    for doc in CORPUS_DOCS
]

print(f"✓ {len(chunks)} chunks ready")
for c in chunks:
    print(f"  [{c['id']}]: {len(c['content'])} chars")

✓ 5 chunks ready
  [billing_cycle]: 454 chars
  [cancellation_policy]: 350 chars
  [refund_policy]: 345 chars
  [plan_changes]: 433 chars
  [payment_methods]: 359 chars


## 4. Create Vector Store and Retriever


In [5]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

chroma_client = chromadb.Client()

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name="phase1_rag",
    embedding_function=embedding_fn,
)

collection.add(
    documents=[c["content"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks],
    ids=[c["id"] for c in chunks],
)

def retrieve_context(question: str, k: int = 3) -> str:
    results = collection.query(query_texts=[question], n_results=k)
    return "\n\n".join(results["documents"][0])

print(f"✓ ChromaDB collection ready with {collection.count()} documents")
print("✓ Retriever function defined")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ ChromaDB collection ready with 5 documents
✓ Retriever function defined


## 5. Create LLM (Anthropic Claude)

In [6]:
import anthropic

claude = anthropic.Anthropic()

def call_claude(system_prompt: str, user_message: str) -> str:
    response = claude.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=512,
        temperature=0,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

print(f"✓ Anthropic client ready, model: {ANTHROPIC_MODEL}")

✓ Anthropic client ready, model: claude-haiku-4-5-20251001


## 6. Define RAG Function


In [7]:
def rag_answer(question: str, system_prompt: str) -> dict:
    retrieved_context = retrieve_context(question, k=3)
    user_message = f"Context:\n{retrieved_context}\n\nQuestion:\n{question}"
    generated_answer = call_claude(system_prompt, user_message)
    return {
        "question": question,
        "retrieved_context": retrieved_context,
        "generated_answer": generated_answer,
    }

print("✓ RAG function defined")

✓ RAG function defined


## 7. Define 5 Questions


In [8]:
questions_with_refs = [
    {
        "question": "What is the billing cycle for monthly subscribers?",
        "reference_answer": "Monthly subscribers are billed on the same calendar date each month.",
    },
    {
        "question": "How can a subscriber cancel their TechCorp Pro subscription?",
        "reference_answer": "Subscribers can cancel at any time through the Account Settings page under the Subscription tab.",
    },
    {
        "question": "What is the refund policy for new subscriptions?",
        "reference_answer": "New subscriptions have a 14-day money-back guarantee. Refund requests must be submitted within 14 days of the initial purchase date.",
    },
    {
        "question": "When does a plan upgrade take effect?",
        "reference_answer": "Upgrades take effect immediately, and the subscriber is charged a prorated amount for the remainder of the current billing cycle.",
    },
    {
        "question": "What payment methods does TechCorp Pro accept?",
        "reference_answer": "TechCorp Pro accepts Visa, Mastercard, American Express, Discover credit/debit cards, and PayPal. Bank transfers (ACH) are available for annual Business or Enterprise plan subscribers.",
    },
]

questions = [q["question"] for q in questions_with_refs]

print(f"✓ {len(questions)} questions defined")
for i, q in enumerate(questions_with_refs, 1):
    print(f"  {i}. {q['question']}")

✓ 5 questions defined
  1. What is the billing cycle for monthly subscribers?
  2. How can a subscriber cancel their TechCorp Pro subscription?
  3. What is the refund policy for new subscriptions?
  4. When does a plan upgrade take effect?
  5. What payment methods does TechCorp Pro accept?


## 8. Define Baseline System Prompt


In [9]:
baseline_system_prompt = (
    "You are a helpful assistant. "
    "Answer only from the provided context. "
    "If the answer is not in the context, say you do not know. "
    "Be concise."
)

print("✓ Baseline system prompt defined")
print(repr(baseline_system_prompt))

✓ Baseline system prompt defined
'You are a helpful assistant. Answer only from the provided context. If the answer is not in the context, say you do not know. Be concise.'


## 9. Run Baseline RAG


In [10]:
baseline_results = []
for item in questions_with_refs:
    result = rag_answer(item["question"], baseline_system_prompt)
    result["reference_answer"] = item["reference_answer"]
    baseline_results.append(result)

baseline_df = pd.DataFrame(baseline_results)[
    ["question", "reference_answer", "retrieved_context", "generated_answer"]
]
baseline_df

,question,reference_answer,retrieved_context,generated_answer
0,What is the billing cycle for monthly subscrib...,Monthly subscribers are billed on the same cal...,TechCorp Pro offers two billing cycles: monthl...,"According to the context, monthly subscribers ..."
1,How can a subscriber cancel their TechCorp Pro...,Subscribers can cancel at any time through the...,Subscribers may cancel at any time through the...,A subscriber can cancel their TechCorp Pro sub...
2,What is the refund policy for new subscriptions?,New subscriptions have a 14-day money-back gua...,TechCorp Pro offers a 14-day money-back guaran...,TechCorp Pro offers a **14-day money-back guar...
3,When does a plan upgrade take effect?,"Upgrades take effect immediately, and the subs...",Subscribers can upgrade their plan at any time...,"According to the context, plan upgrades take e..."
4,What payment methods does TechCorp Pro accept?,"TechCorp Pro accepts Visa, Mastercard, America...","TechCorp Pro accepts Visa, Mastercard, America...",TechCorp Pro accepts the following payment met...


## 10. Manual Baseline Evaluation Table


In [11]:
baseline_manual_eval_df = baseline_df.copy()
baseline_manual_eval_df["manual_quality_label"] = ""
baseline_manual_eval_df["manual_notes"] = ""

baseline_manual_eval_df[
    ["question", "generated_answer", "manual_quality_label", "manual_notes"]
]

,question,generated_answer,manual_quality_label,manual_notes
0,What is the billing cycle for monthly subscrib...,"According to the context, monthly subscribers ...",,
1,How can a subscriber cancel their TechCorp Pro...,A subscriber can cancel their TechCorp Pro sub...,,
2,What is the refund policy for new subscriptions?,TechCorp Pro offers a **14-day money-back guar...,,
3,When does a plan upgrade take effect?,"According to the context, plan upgrades take e...",,
4,What payment methods does TechCorp Pro accept?,TechCorp Pro accepts the following payment met...,,


## 11. Define Degraded System Prompt


In [12]:
degraded_system_prompt = (
    "You are a helpful assistant. "
    "Answer the question using the context if useful. "
    "Add helpful details when appropriate."
)

print("✓ Degraded system prompt defined")
print(repr(degraded_system_prompt))

✓ Degraded system prompt defined
'You are a helpful assistant. Answer the question using the context if useful. Add helpful details when appropriate.'


## 12. Run Degraded RAG


In [13]:
degraded_results = []
for item in questions_with_refs:
    result = rag_answer(item["question"], degraded_system_prompt)
    result["reference_answer"] = item["reference_answer"]
    degraded_results.append(result)

degraded_df = pd.DataFrame(degraded_results)[
    ["question", "reference_answer", "retrieved_context", "generated_answer"]
]
degraded_df

,question,reference_answer,retrieved_context,generated_answer
0,What is the billing cycle for monthly subscrib...,Monthly subscribers are billed on the same cal...,TechCorp Pro offers two billing cycles: monthl...,# Billing Cycle for Monthly Subscribers\n\nMon...
1,How can a subscriber cancel their TechCorp Pro...,Subscribers can cancel at any time through the...,Subscribers may cancel at any time through the...,# How to Cancel Your TechCorp Pro Subscription...
2,What is the refund policy for new subscriptions?,New subscriptions have a 14-day money-back gua...,TechCorp Pro offers a 14-day money-back guaran...,# TechCorp Pro Refund Policy for New Subscript...
3,When does a plan upgrade take effect?,"Upgrades take effect immediately, and the subs...",Subscribers can upgrade their plan at any time...,"According to the context, **plan upgrades take..."
4,What payment methods does TechCorp Pro accept?,"TechCorp Pro accepts Visa, Mastercard, America...","TechCorp Pro accepts Visa, Mastercard, America...",# Payment Methods Accepted by TechCorp Pro\n\n...


## 13. Comparison Table


In [14]:
import re

def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

def exact_match(answer: str, reference: str) -> bool:
    return _normalize(answer) == _normalize(reference)

def contains_reference_keywords(answer: str, reference: str, min_len: int = 4) -> float:
    _stop = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "for",
             "of", "to", "in", "on", "at", "by", "with", "from", "that", "this",
             "it", "its", "be", "been", "have", "has", "not", "can", "will", "do"}
    ref_words = {w for w in re.findall(r"\w+", reference.lower())
                 if len(w) >= min_len and w not in _stop}
    if not ref_words:
        return 0.0
    ans_words = set(re.findall(r"\w+", answer.lower()))
    return round(len(ref_words & ans_words) / len(ref_words), 3)

def jaccard_overlap(text1: str, text2: str) -> float:
    s1 = set(re.findall(r"\w+", text1.lower()))
    s2 = set(re.findall(r"\w+", text2.lower()))
    if not s1 | s2:
        return 0.0
    return round(len(s1 & s2) / len(s1 | s2), 3)

comparison_records = []
for b, d in zip(baseline_results, degraded_results):
    ref = b["reference_answer"]
    b_ans = b["generated_answer"]
    d_ans = d["generated_answer"]
    comparison_records.append({
        "question": b["question"],
        "reference_answer": ref,
        "baseline_answer": b_ans,
        "degraded_answer": d_ans,
        "baseline_context": b["retrieved_context"],
        "degraded_context": d["retrieved_context"],
        "exact_match_baseline": exact_match(b_ans, ref),
        "exact_match_degraded": exact_match(d_ans, ref),
        "baseline_keyword_coverage": contains_reference_keywords(b_ans, ref),
        "degraded_keyword_coverage": contains_reference_keywords(d_ans, ref),
        "answer_length_delta": len(d_ans) - len(b_ans),
        "context_overlap_estimate": jaccard_overlap(b["retrieved_context"], d["retrieved_context"]),
        "notes": "",
    })

comparison_df = pd.DataFrame(comparison_records)

comparison_df[[
    "question",
    "exact_match_baseline", "exact_match_degraded",
    "baseline_keyword_coverage", "degraded_keyword_coverage",
    "answer_length_delta", "context_overlap_estimate",
]]

,question,exact_match_baseline,exact_match_degraded,baseline_keyword_coverage,degraded_keyword_coverage,answer_length_delta,context_overlap_estimate
0,What is the billing cycle for monthly subscrib...,False,False,1.000,1.000,82,1.0
1,How can a subscriber cancel their TechCorp Pro...,False,False,0.556,0.889,597,1.0
2,What is the refund policy for new subscriptions?,False,False,1.000,1.000,331,1.0
3,When does a plan upgrade take effect?,False,False,1.000,0.917,168,1.0
4,What payment methods does TechCorp Pro accept?,False,False,0.789,0.947,210,1.0


## 14. Export Results


In [15]:
baseline_df.to_csv("phase1_baseline_results.csv", index=False)
degraded_df.to_csv("phase1_degraded_results.csv", index=False)
comparison_df.to_csv("phase1_comparison_results.csv", index=False)

print("✓ Exported:")
print("  phase1_baseline_results.csv")
print("  phase1_degraded_results.csv")
print("  phase1_comparison_results.csv")

✓ Exported:
  phase1_baseline_results.csv
  phase1_degraded_results.csv
  phase1_comparison_results.csv


## 15. Why Manual "Vibe Checks" Fail

Manual inspection of LLM outputs is a natural first step, but it cannot function as a reliable evaluation strategy for four reasons:

1. **Subjectivity** — Two reviewers reading the same answer may reach different quality judgments. Without a rubric, quality labels are noisy and inconsistent across reviewers.
2. **Reviewer inconsistency** — A single reviewer is inconsistent across a large dataset: answers reviewed in the morning feel different from those reviewed after fatigue sets in.
3. **No scalability** — Reviewing 5 examples takes minutes. Reviewing 500 examples takes hours. Reviewing 50,000 is humanly impossible. A regression that degrades 5% of answers across 10,000 queries would require 500 careful manual reads just to detect.
4. **Cannot detect small regressions** — A 10% quality degradation caused by a subtly worse system prompt (like the degraded prompt above) is invisible in a 5-example spot-check. You need systematic measurement to catch it reliably.

The comparison table above illustrates this: `answer_length_delta` and `keyword_coverage` hint at differences between baseline and degraded answers, but human eyes alone cannot trust these patterns without running them at scale.

## 16. Why Traditional Unit Testing Fails for LLM Outputs

Classical unit testing relies on deterministic equality: `assert output == expected`. This breaks for LLMs in three ways:

1. **Non-determinism** — Even at `temperature=0`, minor changes to model weights, batching order, or hardware can change exact phrasing without changing meaning. Tests that pass today may silently fail after a model update.
2. **Semantic equivalence with different surface forms** — *"You can cancel via Account Settings"* and *"Cancellation is available under the Subscription tab in Account Settings"* convey the same fact. An `assert x == y` test fails both answers even though both are correct.
3. **Reference answers are not unique** — For most factual questions, many valid phrasings exist. Locking tests to a single reference string creates brittle tests that break on every model update, prompt tweak, or rephrasing.

The `exact_match` column in the comparison table above will almost always be `False` — not because answers are wrong, but because generative models rephrase freely. This makes naive exact-match testing an unreliable quality signal for generative AI.

## 17. What Phase 1 Proves

Phase 1 demonstrates the **Vibe Check Fallacy**:

- Manual inspection of 5 examples can reveal **obvious catastrophic failures** (completely wrong answers, hallucinated facts that contradict the corpus).
- It **cannot** reliably detect **partial regressions**: a subtly worse system prompt that occasionally adds unrequested speculation, reduces grounding by 15%, or causes the model to answer from general knowledge instead of the context.
- The automated helpers (`exact_match`, `keyword_coverage`, `answer_length_delta`, `context_overlap_estimate`) show that numeric signals exist — but interpreting them correctly, calibrating thresholds, and aggregating them across hundreds of questions requires a principled evaluation framework.

**Conclusion:** To move beyond vibe checks, we need structured evaluation metrics — faithfulness, answer relevance, context precision — that can be computed automatically at scale. That is the subject of Phase 2 (Ragas evaluation).

In [ ]:
## Phase 1 Completion Checklist
checklist = {
    "Basic RAG pipeline implemented (ChromaDB + sentence-transformers + Claude)": True,
    "Synthetic corpus created (TechCorp Pro Billing FAQ, 5 documents)": True,
    "5 questions with reference answers defined": len(questions_with_refs) == 5,
    "Baseline prompt run (grounded, concise)": len(baseline_results) == 5,
    "Degraded prompt run (open-ended, allows extra details)": len(degraded_results) == 5,
    "phase1_baseline_results.csv generated": True,
    "phase1_degraded_results.csv generated": True,
    "phase1_comparison_results.csv generated": True,
    "Automated helpers added (exact_match, keyword_coverage, length_delta, context_overlap)": True,
    "Manual-evaluation limitations explained (§15)": True,
    "Unit-testing limitations explained (§16)": True,
}

print("=" * 60)
print("Phase 1 Checklist")
print("=" * 60)
all_passed = True
for item, status in checklist.items():
    icon = "✓" if status else "✗"
    print(f"  {icon} {item}")
    if not status:
        all_passed = False

print()
print("All checks passed ✓" if all_passed else "Some checks FAILED ✗")